
# BigQuery Fundamentals — Employee Dataset

An org wants its HR data in BigQuery — headcount by department, salary by gender/state,
attendance trends — and wants it to stay fast and cheap as attendance history grows into the
millions of rows.

## The data
Three CSVs on GitHub — `departments` (8 rows), `employees` (100 rows), `attendance_monthly`
(1,200 rows: 100 employees × 12 months)

Source: `github.com/accordion-gcp/resources/tree/master/DAY-1`

| Table | Key columns |
|---|---|
| `departments` | `department_id`, `department_name`, `location`, `department_head` |
| `employees` | `employee_id`, name/gender/state/city, `department_id`, `designation`, `date_of_joining`, `monthly_salary_inr` |
| `attendance_monthly` | `employee_id`, `department_id`/`state`, `month_date`, `days_present`, `leaves_taken`, `performance_rating` |



## 1. Auth

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("authenticated")



## 2. Config + a `hr_analytics` dataset to hold everything

In [ ]:
PROJECT_ID = input("Enter GCP Project ID: ").strip()
DATASET_ID = "hr_analytics"

!gcloud config set project {PROJECT_ID}
!gcloud services enable bigquery.googleapis.com

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)

def table(name):
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

client.create_dataset(
    f"{PROJECT_ID}.{DATASET_ID}",
    exists_ok=True
)

print("Dataset ready.")


> 🖥️ SQL workspace should show the new `hr_analytics` dataset, empty for now.



## 3. Pull the CSVs straight from GitHub
`pandas.read_csv` on the raw URL, then `load_table_from_dataframe` — schema gets inferred from the
DataFrame's dtypes. Fine for clean files like these; for anything production-bound, declare an
explicit schema instead so one malformed row can't quietly change a column's type.


In [ ]:
GITHUB_BASE = "https://raw.githubusercontent.com/accordion-gcp/resources/main/DAY-1/"

employees_df = pd.read_csv(f"{GITHUB_BASE}/employees.csv")
departments_df = pd.read_csv(f"{GITHUB_BASE}/departments.csv")
attendance_df = pd.read_csv(f"{GITHUB_BASE}/attendance_monthly.csv")

attendance_df["month_date"] = pd.to_datetime(
    attendance_df["month_date"]
).dt.date

In [ ]:
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

client.load_table_from_dataframe(
    employees_df,
    table("employees").replace("`",""),
    job_config=job_config
).result()
client.load_table_from_dataframe(
    departments_df,
    table("departments").replace("`",""),
    job_config=job_config
).result()
client.load_table_from_dataframe(
    attendance_df,
    table("attendance").replace("`",""),
    job_config=job_config
).result()


> 🖥️ `hr_analytics` should now list all three tables — click one and check the Schema tab against
> what got inferred, and Preview for the actual rows.


In [ ]:
def run_query(sql):
    return client.query(sql).to_dataframe()


## 4. Basic queries
"Karnataka employees earning over ₹80,000, highest first" — filter, sort, done.


In [ ]:
run_query(f"""
SELECT *
FROM `{PROJECT_ID}.hr_analytics.employees`
LIMIT 10
""")

In [ ]:
run_query(f'''
SELECT employee_id, first_name, last_name, state, designation, monthly_salary_inr
FROM `{PROJECT_ID}.hr_analytics.employees`
WHERE state = 'Karnataka' AND monthly_salary_inr > 80000
ORDER BY monthly_salary_inr DESC
''')


In [ ]:
run_query(f"""
SELECT
    COUNT(*) AS total_employees,
    ROUND(AVG(monthly_salary_inr), 2) AS avg_salary,
    MIN(monthly_salary_inr) AS min_salary,
    MAX(monthly_salary_inr) AS max_salary
FROM {table("employees")}
""")



In [ ]:
run_query(f"""
SELECT
    state,
    COUNT(*) AS headcount,
    ROUND(AVG(monthly_salary_inr), 2) AS avg_salary
FROM {table("employees")}
GROUP BY state
ORDER BY headcount DESC
""")


> 🖥️ Run either in the SQL workspace, then click a column header in the results grid to re-sort
> without re-running the query.



## 5. Joins
Employees only have a `department_id` number — the readable name lives in `departments`.


In [ ]:
run_query(f'''
SELECT e.employee_id, e.first_name, e.last_name, d.department_name, d.location
FROM {table("employees")} AS e
INNER JOIN {table("departments")} AS d ON e.department_id = d.department_id
LIMIT 20
''')


In [ ]:
run_query(f'''
SELECT d.department_name, d.location, COUNT(e.employee_id) AS headcount,
       ROUND(AVG(e.monthly_salary_inr), 2) AS avg_salary
FROM {table("departments")} AS d
INNER JOIN {table("employees")} AS e ON d.department_id = e.department_id
GROUP BY d.department_name, d.location
ORDER BY avg_salary DESC
''')



## 6. Views
Stores the query, not the data — every read re-runs the join+aggregation against live tables.


In [ ]:
run_query(f'''
CREATE OR REPLACE VIEW {TBL("department_summary_view")} AS
SELECT d.department_name, d.location, COUNT(e.employee_id) AS headcount,
       ROUND(AVG(e.monthly_salary_inr), 2) AS avg_salary
FROM {table("departments")} AS d
INNER JOIN {table("employees")} AS e ON d.department_id = e.department_id
GROUP BY d.department_name, d.location
''')

run_query(f'SELECT * FROM {table("department_summary_view")} ORDER BY avg_salary DESC')



> 🖥️ `department_summary_view` shows a "view" icon in the Explorer; its Details tab shows
> Table type: VIEW and the stored SQL — no rows of its own.

Always live, zero storage cost, zero performance win — every read redoes the whole join.


## 7. Materialized views

In [ ]:
client.query(f"""
CREATE MATERIALIZED VIEW {table("monthly_attendance_summary_mv")} AS
SELECT
    month_date,
    department_id,
    AVG(days_present) AS avg_days_present,
    AVG(performance_rating) AS avg_performance_rating,
    COUNT(*) AS employee_month_records
FROM {table("attendance")}
GROUP BY month_date, department_id
""").result()

print("Materialized view created.")


In [ ]:
run_query(f"""
SELECT
    month_date,
    department_id,
    ROUND(avg_days_present, 2) AS avg_days_present,
    ROUND(avg_performance_rating, 2) AS avg_performance_rating,
    employee_month_records
FROM {table("monthly_attendance_summary_mv")}
WHERE month_date = '2025-06-01'
ORDER BY avg_performance_rating DESC
""")


> 🖥️ Distinct "materialized view" icon; Details tab has a Refresh section a plain view never
> shows, since it has nothing to refresh.





## 8. Partitioning
`attendance_monthly` is 1,200 rows here — trivial either way. Imagine the same table at 5 years and
10,000 employees: millions of rows, and a "just June" query shouldn't have to touch the rest.


In [ ]:
run_query(f'''
CREATE TABLE {table("attendance_monthly_partitioned")}
PARTITION BY month_date AS
SELECT * FROM {TBL("attendance")}
''')


In [ ]:
print("Unpartitioned Table")
run_query(f"""
SELECT AVG(days_present) AS avg_days_present
FROM {table("attendance")}
WHERE month_date = '2025-06-01'
""")



In [ ]:
print("Partitioned Table")
run_query(f"""
SELECT AVG(days_present) AS avg_days_present
FROM {table("attendance_monthly_partitioned")}
WHERE month_date = '2025-06-01'
""")


## 9. Clustering
Partitioning gets you to a month. Clustering sorts *within* each partition so a department filter
can skip blocks too.


In [ ]:
run_query(f'''
CREATE TABLE {table("attendance_monthly_partitioned_clustered")}
PARTITION BY month_date
CLUSTER BY department_id AS
SELECT * FROM {table("attendance")}
''')


In [ ]:
run_query(f'''
SELECT AVG(performance_rating) FROM {table("attendance_monthly_partitioned")}
WHERE month_date = '2025-06-01' AND department_id = 101
''')




In [ ]:
run_query(f'''
SELECT AVG(performance_rating) FROM {table("attendance_monthly_partitioned_clustered")}
WHERE month_date = '2025-06-01' AND department_id = 101
''')


> 🖥️ The clustered table's Details tab shows both a Partition section and a Clustering section —
> the one place you see both together.

Clustering's payoff doesn't always show cleanly in a dry-run estimate the way partition pruning
does. For a real answer, run both for real and compare Bytes billed, not the estimate.


## Cleanup

In [ ]:
CONFIRM_DELETE = True



In [ ]:
if CONFIRM_DELETE:
    bq_client.delete_dataset(f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True)
    print(f"{DATASET_ID} and everything in it: gone")
